In [1]:
import json
file = '/home/loci/main/tandem_website_dev/tandem/jobs/test/test_website/user_log.jsonl'
events = []
with open(file, "r", encoding="utf-8") as handle:
    for line in handle:
        line = line.strip()
        if not line:
            continue
        try:
            events.append(json.loads(line))
        except json.JSONDecodeError:
            continue

In [ ]:
import html

import gradio as gr


def get_popup_css():
    return """
<style>
  .status-demo {
    background: #111;
    color: #f2f2f2;
    padding: 24px;
    border-radius: 8px;
    font-family: Arial, Helvetica, sans-serif;
    font-size: 28px;
  }

  .popup-link {
    color: #ffd166;
    cursor: pointer;
    text-decoration: underline;
  }

  .popup-overlay {
    display: none;
    position: fixed;
    inset: 0;
    background: rgba(0, 0, 0, 0.72);
    z-index: 9999;
    align-items: center;
    justify-content: center;
    padding: 16px;
  }

  .popup-overlay.is-open {
    display: flex;
  }

  .popup-modal {
    width: min(680px, calc(100vw - 32px));
    background: #191919;
    color: #f2f2f2;
    border: 1px solid #3a3a3a;
    border-radius: 8px;
    overflow: hidden;
  }

  .popup-modal-header,
  .popup-modal-body {
    padding: 18px 20px;
  }

  .popup-modal-header {
    border-bottom: 1px solid #2f2f2f;
    display: flex;
    justify-content: space-between;
    align-items: center;
    gap: 16px;
  }

  .popup-modal-title {
    margin: 0;
    font-size: 1.1rem;
  }

  .popup-modal-close {
    background: #2a2a2a;
    color: #f2f2f2;
    border: 1px solid #404040;
    border-radius: 6px;
    padding: 6px 10px;
    cursor: pointer;
  }

  .popup-modal-body p {
    margin-top: 0;
    line-height: 1.55;
    font-size: 16px;
  }

  .popup-modal-body p:last-child {
    margin-bottom: 0;
  }
</style>
"""
from . import js


def build_popup_trigger(modal_id, trigger_text):
    return '<span class="popup-link" data-open-modal="{}">{}</span>'.format(
        html.escape(modal_id, quote=True),
        html.escape(trigger_text),
    )


def build_popup_modal(modal_id, title, paragraphs):
    body_html = "\n".join(
        "<p>{}</p>".format(html.escape(paragraph)) for paragraph in paragraphs
    )
    return """
<div id="{modal_id}" class="popup-overlay">
  <div class="popup-modal">
    <div class="popup-modal-header">
      <h3 class="popup-modal-title">{title}</h3>
      <button type="button" class="popup-modal-close" data-close-modal="{modal_id}">Close</button>
    </div>
    <div class="popup-modal-body">
      {body_html}
    </div>
  </div>
</div>
""".format(
        modal_id=html.escape(modal_id, quote=True),
        title=html.escape(title),
        body_html=body_html,
    )


def build_popup_text(status_text, modal_id, trigger_text, title, paragraphs):
    trigger_html = build_popup_trigger(modal_id, trigger_text)
    modal_html = build_popup_modal(modal_id, title, paragraphs)
    status_html = '<div class="status-demo">{} {}</div>'.format(
        html.escape(status_text),
        trigger_html,
    )
    return get_popup_css() + status_html + modal_html


rows = []
modals = []

warning_1_trigger = build_popup_trigger("warning-stage-2", "1 Warning")
warning_1_modal = build_popup_modal(
    "warning-stage-2",
    "Feature calculation: 1 warning",
    [
        "Some features could not be calculated for all SAVs.",
        "Please check features.txt and log.txt.",
    ],
)
modals.append(warning_1_modal)

warning_2_trigger = build_popup_trigger("warning-stage-4", "2 Warnings")
warning_2_modal = build_popup_modal(
    "warning-stage-4",
    "Model inferencing: 2 warnings",
    [
        "Some SAVs were skipped during prediction.",
        "Please check Main_Predictions.txt and log.txt.",
    ],
)
modals.append(warning_2_modal)

rows.append(f"""
<tr>
  <td>Done {warning_1_trigger}</td>
  <td><a href="#">features</a></td>
  <td>4.0 min</td>
  <td>Feature calculation</td>
</tr>
""")

rows.append(f"""
<tr>
  <td>Done {warning_2_trigger}</td>
  <td><a href="#">Main_Predictions</a></td>
  <td>13.7 s</td>
  <td>Model inferencing/Training</td>
</tr>
""")

html_text = f"""
{get_popup_css()}

<table>
  <tbody>
    {''.join(rows)}
  </tbody>
</table>

{''.join(modals)}
"""



with gr.Blocks() as demo:
    gr.HTML(value=html_text, js_on_load=js.popup_modal)

demo.launch(server_port=7890)

* Running on local URL:  http://127.0.0.1:7890
* To create a public link, set `share=True` in `launch()`.


In [20]:
demo.close()

In [ ]:
<button class="ps-warning-link" data-dialog-id="feature-warning-dialog" data-bound-open="1">1 Warning</button>